## 2. Understanding questions

1. State the two independence assumptions precisely. Which one fails if a customer's action depends on what they did three steps ago?
2. Why is $\alpha_t(i)$ a *joint* probability but $\beta_t(i)$ a *conditional* one? What breaks if you conflate them?
3. Derive the forward recursion from the definition, naming which assumption licenses each factor.
4. Why is $P(x_{1:T})=\sum_i\alpha_t(i)\beta_t(i)$ true for any $t$? Verify at $t=T$.
5. Forward is $O(Th^2)$ and naive enumeration $O(h^T)$. What structural property makes the saving possible?
6. Why can't you use the $c_t$ scaling trick in Viterbi? What do you do instead?
7. Give a concrete case where the Viterbi path differs from the sequence of individually most-probable states.
8. In Baum–Welch, why does $A_{ij}=\sum_t\xi_t(i,j)/\sum_t\gamma_t(i)$ have that specific denominator? Interpret both sums in words.
9. Why does EM only find a local optimum, and what do you do about it in practice?
10. What is label-switching, and why does it mean you shouldn't interpret "state 3" across two independent training runs?
11. Derive the dwell-time distribution of state $i$. Why is it a limitation?
12. How do you choose $h$? Name two principled approaches.
13. (MC drill) Which is true? (a) Viterbi computes $P(x\mid\lambda)$; (b) Baum–Welch requires labelled states; (c) forward computes the likelihood by summing over all paths; (d) $\beta_t$ is needed for evaluation.
14. You have 10,000 short sequences rather than one long one. What changes in forward and in the M-step?

## Answers to Understanding questions

1. Markov property fails, cause the next tate should be dependent only on the ccurrent one
2. 

In [3]:
import numpy as np

def validate(A, B, pi, tol=1e-8):
    h = A.shape[0]

    # dimensions
    if A.shape != (h, h):        return False
    if B.ndim != 2 or B.shape[0] != h: return False
    if pi.shape != (h,):         return False

    # non-negativity
    if (A < 0).any() or (B < 0).any() or (pi < 0).any(): return False

    # normalisation
    if not np.allclose(A.sum(axis=1), 1, atol=tol):  return False
    if not np.allclose(B.sum(axis=1), 1, atol=tol):  return False
    if not np.isclose(pi.sum(), 1, atol=tol):        return False

    return True

In [5]:
def forward(A, B, pi, obs):
    T = len(obs)
    h = A.shape[0]
    alpha = np.zeros((T, h))

    for i in range(h):                        # init row 0
        alpha[0, i] = pi[i] * B[i, obs[0]]

    for t in range(1, T):                     # time
        for j in range(h):                    # destination state
            total = 0.0
            for i in range(h):                # source state (the sum)
                total += alpha[t-1, i] * A[i, j]
            alpha[t, j] = total * B[j, obs[t]]

    return alpha, alpha[-1].sum()

In [6]:
import numpy as np

A  = np.array([[0.8, 0.2],
               [0.4, 0.6]])          # rows: sunny, rainy
B  = np.array([[0.7, 0.2, 0.1],      # sunny -> dry, damp, soggy
               [0.1, 0.3, 0.6]])     # rainy -> dry, damp, soggy
pi = np.array([0.6, 0.4])

obs = np.array([0, 2])               # dry, soggy

alpha, p = forward(A, B, pi, obs)
print(alpha)
print("P(x) =", p)

[[0.42   0.04  ]
 [0.0352 0.0648]]
P(x) = 0.1
